# HP 8673H control demo

Exercises the `HP8673H` class in `hp8673h.py`. Run cells one at a time and watch the generator's front panel display/annunciators to confirm each command took effect.

In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
rm.list_resources()

Find the generator's GPIB address in the list above (factory default is `GPIB0::19::INSTR`) and update the resource string below if it differs.

In [ ]:
from hp8673h import HP8673H

gen = HP8673H("GPIB0::19::INSTR", debug=True)

## Preset and basic CW control

In [ ]:
gen.preset()  # equivalent to pressing RCL 0 on the front panel

In [ ]:
gen.set_frequency_hz(3.0e9)  # 3.000000 GHz
gen.set_power_dbm(-10)
gen.auto_peak(True)
gen.rf_on()

Read back the currently displayed parameter (addresses the generator to talk).

Should print the CW frequency in MHz, e.g. `3000000000.00`, matching what's shown in the FREQUENCY MHz display.

In [ ]:
gen.read_active_parameter()

## Step through a few CW frequencies

Simple software-controlled sweep: set frequency, wait, move on. This is the approach to use later when synchronizing each point with a spectrum analyzer measurement.

In [ ]:
import numpy as np
import time

freqs_hz = np.linspace(2.0e9, 4.0e9, 11)

for f in freqs_hz:
    gen.set_frequency_hz(f)
    time.sleep(0.1)
    print(f"set {f/1e9:.3f} GHz -> readback {gen.read_active_parameter()}")

## Hardware AUTO sweep

Instead of stepping from Python, let the generator run its own internal sweep. Useful for a quick visual check on a scope/analyzer, or for driving an analyzer's sweep from the rear-panel SWP OUT ramp.

In [ ]:
gen.setup_sweep(start_hz=2.0e9, stop_hz=9.0e9, num_steps=50, dwell_ms=20)
gen.start_auto_sweep()

In [ ]:
gen.stop_sweep()

## Single sweep (one-shot, useful once triggering the spectrum analyzer per-sweep)

In [ ]:
gen.setup_sweep(start_hz=2.0e9, stop_hz=9.0e9, num_steps=50, dwell_ms=20)
gen.start_single_sweep()

## Clean up

In [ ]:
gen.rf_off()
gen.go_to_local()
gen.close()